# AI Smart Energy Audit - 4 Case Studies Analysis
This notebook evaluates the Isolation Forest (Anomaly Detection) and Random Forest (Power-Delta Prediction) models across four distinct grid scenarios:
1. **Low Voltage**
2. **Normal Grid Supply**
3. **Demand = Supply** (Net Zero)
4. **PVe + Normal Grid**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)

# ── Build & train Isolation Forest inline so the scaler
# ── is calibrated to the same feature distribution used in production.
try:
    df_train = pd.read_csv('raw_data.csv')
    dft = df_train.copy()
    dft['delta_v'] = dft['voltage'].diff().fillna(0)
    dft['delta_i'] = dft['current'].diff().fillna(0)
    dft['delta_p'] = dft['power'].diff().fillna(0)
    dft['hist_mean_p_10'] = dft['power'].rolling(10, min_periods=1).mean()
    dft['hist_mean_p_50'] = dft['power'].rolling(50, min_periods=1).mean()
    X_train = dft[['delta_v','delta_i','delta_p','hist_mean_p_10','hist_mean_p_50']].values

    iso_scaler = StandardScaler()
    X_train_scaled = iso_scaler.fit_transform(X_train)

    iso_model = IsolationForest(n_estimators=200, contamination=0.02, random_state=42, n_jobs=-1)
    iso_model.fit(X_train_scaled)
    print(f'✅ Isolation Forest trained on {len(X_train):,} samples (contamination=2%).')
except Exception as e:
    iso_model, iso_scaler = None, None
    print(f'❌ Training failed: {e}')



In [ ]:
def evaluate_and_plot(case_name, df):
    print(f'\n========== {case_name} ==========')
    dfd = df.copy().reset_index(drop=True)
    dfd['delta_v']        = dfd['voltage'].diff().fillna(0)
    dfd['delta_i']        = dfd['current'].diff().fillna(0)
    dfd['delta_p']        = dfd['power'].diff().fillna(0)
    dfd['hist_mean_p_10'] = dfd['power'].rolling(10, min_periods=1).mean()
    dfd['hist_mean_p_50'] = dfd['power'].rolling(50, min_periods=1).mean()
    features = dfd[['delta_v','delta_i','delta_p','hist_mean_p_10','hist_mean_p_50']].values

    anomalies = np.zeros(len(dfd), dtype=bool)
    if iso_model is not None:
        try:
            iso_scaled = iso_scaler.transform(features)
            preds = iso_model.predict(iso_scaled)
            anomalies = (preds == -1)
            print(f'  → {anomalies.sum()} anomaly points detected ({anomalies.mean()*100:.1f}%)')
        except Exception as e:
            print(f'  ⚠️  Inference error: {e}')
    dfd['is_anomaly'] = anomalies
    idx = dfd.index

    # ── Clean line-chart style ──
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(14, 10))
    fig.suptitle(f'{case_name}', fontsize=15, fontweight='bold', y=1.01)
    styles = [
       ('#4a90d9', 'Voltage (V)',    'voltage'),
       ('#e67e22', 'Current (A)',    'current'),
       ('#27ae60', 'Real Power (W)', 'power'),
    ]
    for ax, (color, ylabel, col) in zip(axes, styles):
        ax.plot(idx, dfd[col], color=color, lw=1.4, alpha=0.9, label=ylabel)
        # Mark anomaly points with red stars only
        adf = dfd[dfd['is_anomaly']]
        if not adf.empty:
            ax.scatter(adf.index, adf[col],
                       marker='*', s=200, color='#e74c3c',
                       zorder=5, label='Anomaly', edgecolors='#c0392b', linewidths=0.5)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.legend(loc='upper right', fontsize=9)
    axes[2].set_xlabel('Sample Index', fontsize=10)
    plt.tight_layout()
    safe = case_name.replace(' ', '_').replace(':', '').replace('=', '_')
    plt.savefig(f'{safe}.png', dpi=150, bbox_inches='tight')
    plt.show()
    return dfd



### Case 1: Low Voltage
Simulating a grid that is under strain, consistently supplying around 200V with intermittent severe drops. Let's capture how models perceive the rapid shift.


In [ ]:
np.random.seed(42)
n = 150
t = np.linspace(0, 10, n)
# Tight stable baseline — small noise so normal readings cluster tightly
v_low = 200 + 0.5*np.sin(t) + np.random.normal(0, 0.08, n)
i_low = 5.0 + 0.1*np.cos(t) + np.random.normal(0, 0.02, n)
# Single clear voltage sag event
v_low[60:72] = np.random.normal(158, 1.5, 12)
p_low = v_low * i_low
df_case1 = pd.DataFrame({'voltage': v_low, 'current': i_low, 'power': p_low})
res1 = evaluate_and_plot('Case 1: Low Voltage Grid', df_case1)


### Case 2: Normal Grid Supply
Standard household 230V mains with typical residential load cycles. A heavy load (e.g., compressor / air conditioner) suddenly engages.


In [ ]:
np.random.seed(43)
n = 150
t = np.linspace(0, 10, n)
# Stable 230V supply with very low noise
v_norm = 230 + 0.3*np.sin(t) + np.random.normal(0, 0.08, n)
i_norm = 3.0 + 0.1*np.cos(t) + np.random.normal(0, 0.02, n)
# Air-conditioner compressor kicks in — clean step change
i_norm[80:120] += 9.0
p_norm = v_norm * i_norm
df_case2 = pd.DataFrame({'voltage': v_norm, 'current': i_norm, 'power': p_norm})
res2 = evaluate_and_plot('Case 2: Normal Grid Supply', df_case2)


### Case 3: Demand = Supply
A steady state (Microgrid/Net Zero environment) where local generation (battery/solar) perfectly balances household demand. Meaning the net current off the main external supply effectively floats at zero Watts, barring very minor mismatches.


In [ ]:
np.random.seed(44)
n = 150
t = np.linspace(0, 10, n)
# Very stable near-zero current (near-perfect balance)
v_bal = 230 + 0.2*np.sin(t) + np.random.normal(0, 0.05, n)
i_bal = np.random.normal(0, 0.03, n)
# Two sharp mismatch spikes — clearly visible anomalies
i_bal[55:58]   = np.random.normal(11.0, 0.3, 3)
i_bal[115:118] = np.random.normal(-9.0, 0.3, 3)
p_bal = v_bal * i_bal
df_case3 = pd.DataFrame({'voltage': v_bal, 'current': i_bal, 'power': p_bal})
res3 = evaluate_and_plot('Case 3: Demand = Supply', df_case3)


### Case 4: PVe + Normal Grid
High solar penetration creating a 'Duck Curve'. During mid-day (solar peak), home demand drops while PV pushes massive power back into the grid (negative net load). Local grid voltage naturally rises due to power export. Intense cloud blocks suddenly kill PV capacity abruptly bringing net load massively positive (returning to import).


In [ ]:
np.random.seed(45)
n = 150
x = np.linspace(-1, 1, n)
# Stable duck-curve demand (low noise)
house_demand = 5 + 3*(x**2) + np.random.normal(0, 0.05, n)
# Smooth solar bell curve
pv_gen = 14 * np.exp(-(x*2.5)**2)
# Two distinct cloud-block events — sharp drops in generation
pv_gen[60:70]  *= 0.05   # near-total loss
pv_gen[115:125] *= 0.10  # severe loss
i_pve = house_demand - pv_gen
v_pve = 230 - (0.5 * i_pve) + np.random.normal(0, 0.08, n)
p_pve = v_pve * i_pve
df_case4 = pd.DataFrame({'voltage': v_pve, 'current': i_pve, 'power': p_pve})
res4 = evaluate_and_plot('Case 4: PVe + Normal Grid', df_case4)


### Conclusion: All Cases Combined
A single visual summary chart comprising all four cases across voltage, current, and real power dimensions.


In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 16))
fig.suptitle('Comprehensive Energy Audit — 4 Case Studies', fontsize=18, fontweight='bold', y=1.01)

datasets = [
    ('Case 1: Low Voltage',   res1),
    ('Case 2: Normal Grid',   res2),
    ('Case 3: Demand=Supply', res3),
    ('Case 4: PVe+Grid',      res4)
]

cols_info = [('voltage','#4a90d9','Volts'), ('current','#e67e22','Amps'), ('power','#27ae60','Watts')]

for i, (name, df_case) in enumerate(datasets):
    for j, (col, color, unit) in enumerate(cols_info):
        ax = axes[i, j]
        ax.plot(df_case.index, df_case[col], color=color, lw=1.2, alpha=0.9)
        adf = df_case[df_case['is_anomaly']]
        if not adf.empty:
            ax.scatter(adf.index, adf[col], marker='*', s=140,
                       color='#e74c3c', zorder=5)
        ax.set_title(f'{name} — {col.capitalize()}', fontsize=9, fontweight='bold')
        ax.set_ylabel(unit, fontsize=8)
        ax.grid(True, alpha=0.2, linestyle='--')

plt.tight_layout()
plt.savefig('combined_case_studies_summary.png', dpi=180, bbox_inches='tight')
plt.show()
